In [15]:
import os
import certifi
from pymongo import MongoClient
from pymongo.server_api import ServerApi
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Get MongoDB credentials from environment variables
MONGO_URI = os.getenv("MONGO_URI")
DB_NAME = os.getenv("MONGO_DB_NAME")
ARTICLES_COLLECTION_NAME = os.getenv("ARTICLES_COLLECTION_NAME")

# Ensure all necessary environment variables are set
if not all([MONGO_URI, DB_NAME, ARTICLES_COLLECTION_NAME]):
    print("❌ Missing required environment variables. Check your .env file.")
    exit()



# Create MongoDB client with TLS verification
try:
    client = MongoClient(MONGO_URI, server_api=ServerApi('1'), tlsCAFile=certifi.where())
    db = client[DB_NAME]
    collection = db[ARTICLES_COLLECTION_NAME]

    # Verify connection
    client.admin.command('ping')
    print("✅ Connected to MongoDB Atlas!")

except Exception as e:
    print(f"❌ MongoDB Connection Error: {e}")
    exit()

Python-dotenv could not parse statement starting at line 1
Python-dotenv could not parse statement starting at line 2
Python-dotenv could not parse statement starting at line 3
Python-dotenv could not parse statement starting at line 4
Python-dotenv could not parse statement starting at line 5
Python-dotenv could not parse statement starting at line 6
Python-dotenv could not parse statement starting at line 7


✅ Connected to MongoDB Atlas!


In [18]:
from datetime import datetime, timezone

# Set the expected date format (e.g., "22-01-2016")
date_format = "%d-%m-%Y"

print("🔍 Starting update of documents with meta.date as string...")

updated_count = 0
skipped_count = 0

cursor = collection.find({
    "meta.date": {
        "$exists": True,
        "$type": "string"
    }
}).sort("_id", 1)

for doc in cursor:
    try:
        doc_id = doc["_id"]
        raw_date = doc["meta"]["date"]

        # Parse and convert to ISODate with UTC timezone
        parsed_date = datetime.strptime(raw_date, date_format).replace(tzinfo=timezone.utc)

        result = collection.update_one(
            {"_id": doc_id},
            {"$set": {"meta.date": parsed_date}}
        )

        if result.modified_count > 0:
            updated_count += 1
            print(f"✅ Updated doc ID {doc_id} (date: {raw_date} → {parsed_date.isoformat()})")
        else:
            print(f"ℹ️ No changes made for doc ID {doc_id} (already up to date?)")

    except Exception as e:
        print(f"⚠️ Error processing doc ID {doc.get('_id')}: {e}")
        skipped_count += 1

# ---- SUMMARY ----
print(f"\n📊 Summary:")
print(f"   ✅ Successfully updated: {updated_count}")
print(f"   ⚠️ Skipped due to errors: {skipped_count}")

🔍 Starting update of documents with meta.date as string...
✅ Updated doc ID 67fa98cceaa4fb525afef4a6 (date: 15-02-2024 → 2024-02-15T00:00:00+00:00)
✅ Updated doc ID 67fa98cceaa4fb525afef4a8 (date: 23-03-2023 → 2023-03-23T00:00:00+00:00)
✅ Updated doc ID 67fa98cdeaa4fb525afef4aa (date: 21-12-2021 → 2021-12-21T00:00:00+00:00)
✅ Updated doc ID 67fa98cdeaa4fb525afef4ac (date: 12-01-2018 → 2018-01-12T00:00:00+00:00)
✅ Updated doc ID 67fa98ceeaa4fb525afef4ae (date: 15-05-2018 → 2018-05-15T00:00:00+00:00)
✅ Updated doc ID 67fa98ceeaa4fb525afef4b0 (date: 10-04-2024 → 2024-04-10T00:00:00+00:00)
✅ Updated doc ID 67fa98cfeaa4fb525afef4b2 (date: 18-10-2018 → 2018-10-18T00:00:00+00:00)
✅ Updated doc ID 67fa98cfeaa4fb525afef4b4 (date: 21-03-2017 → 2017-03-21T00:00:00+00:00)
✅ Updated doc ID 67fa98d0eaa4fb525afef4b6 (date: 20-07-2016 → 2016-07-20T00:00:00+00:00)
✅ Updated doc ID 67fa98d0eaa4fb525afef4b8 (date: 23-06-2023 → 2023-06-23T00:00:00+00:00)
✅ Updated doc ID 67fa98d1eaa4fb525afef4ba (date: 03

In [21]:
# Check total count first
total_matching = collection.count_documents({"meta.date": {"$type": "string"}})
print(f"📦 Found {total_matching} documents with string meta.date.")

📦 Found 0 documents with string meta.date.


In [29]:
sample_cursor = collection.find(
    {"meta.date": {"$exists": True}},
    {"_id": 1, "meta": 1}
).limit(5)

In [9]:
count = collection.count_documents({
    "$or": [
        {"meta.date": {"$exists": False}},
        {"meta.date": {"$not": {"$type": "string"}}}
    ]
})
print(f"found {count} documents without meta.date as string")

found 19257 documents without meta.date.


In [10]:
count = collection.count_documents({
    "meta.date": {"$exists": True}
})
print(f"found {count} documents with meta.date.")

found 38353 documents with meta.date.


In [17]:
count = collection.count_documents({
    "meta.date": {
        "$exists": True,
        "$type": "string"
    }
})
print(f"found {count} documents with meta.date is string")

found 4484 documents with meta.date is string


In [28]:
import json
from bson import json_util
for doc in sample_cursor:
    print(json.dumps(doc, indent=2, default=json_util.default))